# Day 11 — K-Means ile Baskın Renk Paleti
## Piksel Kümeleme, K-Means Algoritması ve Jakarlı Halı İplik Paleti Çıkarımı

> **Aşama:** Faz 2 — Bilgisayarlı Görü (Day 09–15)
> **Resmi Staj Defteri Konusu:** K-Means ile Baskın Renk Paleti (Yaprak 21 & 22)

### 1. Problem
Tasarım stüdyosunda hazırlanan yüksek çözünürlüklü halı desenleri milyonlarca farklı RGB renk varyasyonu içerir. Ancak endüstriyel jakarlı dokuma tezgâhları sınırlı sayıda (örneğin 6 ila 12 adet) iplik bobiniyle çalışabilir. Sürekli renk uzayındaki desenlerin tezgâhta dokunabilecek baskın renklere kuantize edilmesi zorunludur.

### 2. Why the Problem Matters
K-Means kümeleme algoritması, görüntüdeki tüm pikselleri renk uzayında k adet ağırlık merkezine (centroid) atayarak desenin algısal kompozisyonunu en az bilgi kaybıyla koruyan baskın iplik paletini çıkarır.

### 3. Engineering Concepts
- **K-Means Algoritması**: En yakın küme merkezine atama (Expectation) ve merkezleri güncelleme (Maximization) adımlarıyla hata kareleri toplamını (inertia) minimize etme.
- **Palet Oranı (Color Proportions)**: Her kümedeki piksel sayısının toplam piksel sayısına oranı.
- **Deterministik Tohum (Seed)**: Üretim tutarlılığı için deterministik kümeleme.

In [ ]:
# 4. Library / API Investigation & Standalone Definitions
from sklearn.cluster import KMeans
import numpy as np
import cv2
from typing import List
from pydantic import BaseModel

class ColorCluster(BaseModel):
    bgr: List[int]
    rgb: List[int]
    proportion_pct: float
    hex_code: str

class PaletteResult(BaseModel):
    k_clusters: int
    clusters: List[ColorCluster]

class KMeansPaletteExtractor:
    def __init__(self, n_colors: int = 4, random_state: int = 42):
        self.n_colors = n_colors
        self.random_state = random_state

    def extract_palette(self, img: np.ndarray) -> PaletteResult:
        pixels = img.reshape(-1, 3).astype(np.float32)
        km = KMeans(n_clusters=self.n_colors, random_state=self.random_state, n_init=10)
        km.fit(pixels)
        counts = np.bincount(km.labels_, minlength=self.n_colors)
        total = len(pixels)
        clusters = []
        for i, center in enumerate(km.cluster_centers_):
            b, g, r = [int(round(c)) for c in center]
            pct = round(float(counts[i] / total * 100.0), 2)
            hex_c = f"#{r:02X}{g:02X}{b:02X}"
            clusters.append(ColorCluster(
                bgr=[b, g, r],
                rgb=[r, g, b],
                proportion_pct=pct,
                hex_code=hex_c
            ))
        clusters.sort(key=lambda x: x.proportion_pct, reverse=True)
        return PaletteResult(k_clusters=len(clusters), clusters=clusters)

extractor = KMeansPaletteExtractor(n_colors=4, random_state=42)
print("K-Means Palet Motoru Başlatıldı.")


In [ ]:
# 5. Minimal Implementation
# 4 bölgeli sentetik halı görseli (Kırmızı, Lacivert, Krem, Altın)
img = np.zeros((200, 200, 3), dtype=np.uint8)
img[:100, :100] = [30, 30, 180]    # Kırmızı (BGR)
img[:100, 100:] = [140, 40, 20]    # Lacivert (BGR)
img[100:, :100] = [210, 220, 230]  # Krem (BGR)
img[100:, 100:] = [30, 160, 210]   # Altın Sarısı (BGR)

result = extractor.extract_palette(img)
print(f"Küme Sayısı: {result.k_clusters}")
for c in result.clusters:
    print(f"  {c.hex_code} | RGB={c.rgb} | Oran=%{c.proportion_pct:.1f}")

In [ ]:
# 6. Experiment: Palet Kuantizasyonu Doğrulaması
total_pct = sum(c.proportion_pct for c in result.clusters)
print(f"Toplam Palet Kapsama Oranı: %{total_pct:.2f}")

In [ ]:
# 7. Visualization: Baskın Renk Paleti Çubuğu
import matplotlib.pyplot as plt

palette_bar = np.zeros((50, 400, 3), dtype=np.uint8)
start_x = 0
for c in result.clusters:
    width = int(round((c.proportion_pct / 100.0) * 400))
    palette_bar[:, start_x:start_x+width] = c.rgb
    start_x += width

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Orijinal Desen")
axes[0].axis("off")

axes[1].imshow(palette_bar)
axes[1].set_title("Çıkarılan Baskın İplik Paleti")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 8. Validation
assert len(result.clusters) == 4
assert abs(total_pct - 100.0) < 1.0
print("K-Means palet çıkarımı ve oran bütünlüğü başarıyla doğrulandı.")

In [ ]:
# 9. Failure Cases: Tek renkli homojen görsel kümeleme
flat_img = np.ones((50, 50, 3), dtype=np.uint8) * 128
flat_res = KMeansPaletteExtractor(n_colors=2, random_state=42).extract_palette(flat_img)
print("Homojen görselde K-Means davranışı doğrulandı:", flat_res.clusters[0].hex_code)

### 10. Conclusions
K-Means kümeleme yöntemiyle tekstil desenlerinden baskın renk paletleri başarıyla çıkarılmış, dokuma tezgâhı iplik sınırlamalarına uygun renk kuantizasyonu sağlanmıştır.